<a href="https://colab.research.google.com/github/vigneshj10x/Codeboosters-Internship-2026/blob/main/DAY_02_Database_SQL_Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
print('all libraries imported successfully!')
print(f'pandas version: {pd.__version__}')
print(f'sqlite version: {sqlite3.version}')

all libraries imported successfully!
pandas version: 2.2.2
sqlite version: 2.6.0


In [ ]:
df=pd.read_csv('student_performance.csv')
print(f'Dataset loaded successfully with shape: {df.shape}')
df.head(3)

Dataset loaded successfully with shape: (30, 13)


,student_id,name,age,gender,department,semester,math_score,science_score,english_score,programming_score,attendance_percentage,city,admission_year
0,1001,Aarav Sharma,19,Male,Computer Science,2,85,78,72,91,92,Mumbai,2023
1,1002,Priya Patel,20,Female,Computer Science,2,76,82,88,79,87,Ahmedabad,2023
2,1003,Rohit Verma,19,Male,Electronics,2,65,74,61,55,78,Delhi,2023


In [ ]:
conn=sqlite3.connect('college.db')
cursor = conn.cursor()

# Drop the table if it already exists (optional, for clean re-runs)
cursor.execute("DROP TABLE IF EXISTS students")

# Create the students table from the DataFrame.columns
df.to_sql('students', conn, if_exists='replace', index=False)

cursor.execute("SELECT COUNT (*) FROM students")
count = cursor.fetchone()[0]
print(f"database 'college.db'created successfully!")
print(f"number of students in the database: {count}")

conn.close()

database 'college.db'created successfully!
number of students in the database: 30


In [ ]:
conn = sqlite3.connect('college.db')
cursor = conn.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS students (
        student_id INTEGER PRIMARY KEY,
        name TEXT,
        age INTEGER,
        gender TEXT,
        department TEXT,
        semester INTEGER,
        math_score INTEGER,
        science_score INTEGER,
        english_score INTEGER,
        programming_score INTEGER,
        attendance_percentage INTEGER,
        city TEXT,
        admission_year INTEGER
    )
''')

# Insert data from DataFrame into the 'students' table
df.to_sql('students', conn, if_exists='replace', index=False)

# Commit the changes to the database
conn.commit()

# Now, the original query from the problematic cell should work
cursor.execute("SELECT COUNT (*) FROM students")
count = cursor.fetchone()[0]
print(f"database 'college.db' created successfully with data!")
print(f"number of students in the database: {count}")

conn.close()

database 'college.db' created successfully with data!
number of students in the database: 30


In [ ]:
conn = sqlite3.connect('college.db')
sql_query = """
SELECT
    department,
    gender,
    AVG(math_score) AS average_math_score
FROM
    students
GROUP BY
    department, gender
HAVING
    AVG(math_score) > 75
ORDER BY
    average_math_score DESC;
"""

# Execute the query and load results into a pandas DataFrame
result_df = pd.read_sql_query(sql_query, conn)

print("Departments and Genders with Average Math Score > 75:")
display(result_df)

conn.close()

Departments and Genders with Average Math Score > 75:


,department,gender,average_math_score
0,Computer Science,Female,87.428571
1,Computer Science,Male,83.500000


In [ ]:
conn = sqlite3.connect('college.db')
sql_query_gender_analysis = """
SELECT
    gender,
    AVG(math_score) AS average_math_score,
    AVG(science_score) AS average_science_score,
    AVG(english_score) AS average_english_score,
    AVG(programming_score) AS average_programming_score
FROM
    students
GROUP BY
    gender
ORDER BY
    gender;
"""

# Execute the query and load results into a pandas DataFrame
gender_analysis_df = pd.read_sql_query(sql_query_gender_analysis, conn)

print("Gender-wise Average Scores:")
display(gender_analysis_df)

conn.close()

Gender-wise Average Scores:


,gender,average_math_score,average_science_score,average_english_score,average_programming_score
0,Female,78.466667,81.200000,80.800000,70.2
1,Male,73.666667,74.466667,67.533333,65.0


In [ ]:
conn = sqlite3.connect('college.db')
sql_query_strong_score_ranking = """
SELECT
    student_id,
    name,
    department,
    gender,
    (math_score + science_score + english_score + programming_score) / 4.0 AS strong_score,
    RANK() OVER (ORDER BY (math_score + science_score + english_score + programming_score) / 4.0 DESC) AS strong_score_rank
FROM
    students
ORDER BY
    strong_score_rank ASC;
"""

# Execute the query and load results into a pandas DataFrame
strong_score_ranking_df = pd.read_sql_query(sql_query_strong_score_ranking, conn)

print("Student Ranking based on 'Strong Score' (Average of all Subjects):")
display(strong_score_ranking_df)

conn.close()

Student Ranking based on 'Strong Score' (Average of all Subjects):


,student_id,name,department,gender,strong_score,strong_score_rank
0,1010,Ananya Das,Computer Science,Female,92.75,1
1,1022,Tanvi Mehta,Computer Science,Female,91.75,2
2,1030,Akanksha Yadav,Computer Science,Female,91.25,3
3,1005,Arjun Nair,Computer Science,Male,89.00,4
4,1008,Divya Singh,Computer Science,Female,89.00,4
5,1018,Swati Kulkarni,Computer Science,Female,88.50,6
6,1025,Amit Bose,Computer Science,Male,84.00,7
7,1013,Suresh Rao,Computer Science,Male,83.50,8
8,1020,Nisha Kapoor,Computer Science,Female,81.75,9
9,1001,Aarav Sharma,Computer Science,Male,81.50,10


In [ ]:
conn = sqlite3.connect('college.db')
cursor = conn.cursor()

# Get unique department names from the students DataFrame
unique_departments = df['department'].unique()

# Create a DataFrame for departments with department_id
departments_data = pd.DataFrame({
    'department_id': range(1, len(unique_departments) + 1),
    'department_name': unique_departments
})

# Create the 'departments' table in the database
cursor.execute('''
    CREATE TABLE IF NOT EXISTS departments (
        department_id INTEGER PRIMARY KEY,
        department_name TEXT UNIQUE
    )
''')

# Insert data from the departments DataFrame into the 'departments' table
departments_data.to_sql('departments', conn, if_exists='replace', index=False)

# Commit the changes to the database
conn.commit()

print("Departments table created and populated successfully:")

# Display the new departments table
display(pd.read_sql_query("SELECT * FROM departments", conn))

conn.close()

Departments table created and populated successfully:


,department_id,department_name
0,1,Computer Science
1,2,Electronics
2,3,Mechanical
3,4,Civil
